In [1]:
import sys; sys.path.append("..")  # project root, so `src` imports work
import os, pandas as pd, numpy as np
from sentence_transformers import SentenceTransformer
from src.db import get_engine
engine = get_engine()

In [2]:
news = pd.read_sql("SELECT news_id, title, abstract FROM dim_news", engine)
news["text"] = news["title"].fillna("") + ". " + news["abstract"].fillna("")
print(len(news), "articles")
news[["news_id", "text"]].head()

51282 articles


,news_id,text
0,N55528,"The Brands Queen Elizabeth, Prince Charles, an..."
1,N19639,50 Worst Habits For Belly Fat. These seemingly...
2,N61837,The Cost of Trump's Aid Freeze in the Trenches...
3,N53526,I Was An NBA Wife. Here's How It Affected My M...
4,N38324,"How to Get Rid of Skin Tags, According to a De..."


In [3]:
model = SentenceTransformer("all-MiniLM-L6-v2")     

emb = model.encode(
    news["text"].tolist(),
    batch_size=128,
    show_progress_bar=True,
    normalize_embeddings=True,      
)
print(emb.shape)                    

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/401 [00:00<?, ?it/s]

(51282, 384)


In [4]:
os.makedirs("../models", exist_ok=True)
np.save("../models/news_emb.npy", emb)
np.save("../models/news_emb_ids.npy", news["news_id"].values)   # row → news_id mapping
print("saved", emb.shape, "embeddings")

saved (51282, 384) embeddings


In [5]:
news_ids = news["news_id"].values
row_of   = {nid: i for i, nid in enumerate(news_ids)}
title_of = dict(zip(news["news_id"], news["title"]))

def nearest(nid, k=5):
    v = emb[row_of[nid]]
    sims = emb @ v                      # dot product = cosine (vectors are normalized)
    top = np.argsort(-sims)[1:k+1]      # [0] is the article itself, so skip it
    return [(title_of[news_ids[i]], float(sims[i])) for i in top]

probe = news_ids[100]
print("QUERY:", title_of[probe])
for t, s in nearest(probe):
    print(f"  {s:.3f}  {t}")

QUERY: Man killed in officer-involved shooting in St. Louis
  0.763  Man shot after firing shots at officers in East St. Louis
  0.733  Man shot and killed in north St. Louis
  0.717  KCPD: Man killed while trying to prevent break-in
  0.715  Man killed in shooting on Pasadena Street in Greensboro; man identified
  0.701  Man injured after shots fired on St. Louis interstate
